# LifeGuard-RL — Late-life SOH forecast from cycle 100

Compare the original digital twin with a forecast calibrated to the earlier remaining useful life (RUL) model. The task is a single forecast issued at cycle 100.

1. Select your Python kernel and run all cells.
2. The notebook finds **lifeguard_digital_twin_validation_results.zip** or its extracted folder under `reports/` and reuses the five saved cell-level splits. Set `INPUT_PATH` if the report is stored elsewhere.
3. Open **lifeguard_fixed_anchor_soh_results.zip** in the input report's parent folder. Set `OUTPUT_ROOT` to choose a different destination.

The forecast uses the held-out cell's charge-based initial SOH and saved RUL prediction. Measured SOH after cycle 100 is used to learn the decay shape on training cells and to score forecasts on held-out cells. This is an exploratory comparison on the same 23 XJTU cells.

In [1]:
from bisect import bisect_left
from collections import defaultdict
from contextlib import ExitStack
import csv
import hashlib
from io import TextIOWrapper
import json
from math import floor, isclose, isfinite
from pathlib import Path, PurePosixPath
from statistics import mean, median
from zipfile import ZIP_DEFLATED, ZipFile

INPUT_PATH = None  # Optional ZIP or extracted Notebook 07 report folder.
OUTPUT_ROOT = None  # Defaults to the input report's parent folder.

INPUT_NAME = "lifeguard_digital_twin_validation_results"
OUTPUT_NAME = "lifeguard_fixed_anchor_soh_results"
ANCHOR = 100
THRESHOLD = 80.0
BASELINE = "soh_dependent"
CANDIDATE = "late_life_calibrated"
EARLIER_RUL = "previous_rul_model"
ORIGINAL_MODELS = ("persistence", "constant_fade", BASELINE, "soh_dependent_with_history")
SOH_MODELS = (*ORIGINAL_MODELS, CANDIDATE)
HORIZONS = (1, 10, 50)
TAIL_CYCLES = 20
HORIZON_TOLERANCE_PP = 0.05
PHASE_GRID = tuple(step / 100 for step in range(101))
EPS = 1e-9


def locate_input():
    if INPUT_PATH is not None:
        return Path(INPUT_PATH).expanduser().resolve(strict=True)
    working_dir = Path.cwd()
    for root in (working_dir, *working_dir.parents):
        report = root / "reports" / INPUT_NAME
        archive = report.with_suffix(".zip")
        if archive.is_file():
            return archive
        if report.is_dir():
            return report
    raise FileNotFoundError(f"Could not find reports/{INPUT_NAME}.zip or its extracted folder. "
                            "Set INPUT_PATH to the ZIP or report folder.")


def open_report(source, stack):
    if source.is_dir():
        return lambda name: (source / name).open(encoding="utf-8", newline="")
    archive = stack.enter_context(ZipFile(source))
    members = [entry for entry in archive.infolist() if not entry.is_dir()]
    configs = [entry for entry in members if PurePosixPath(entry.filename).name == "run_config.json"]
    if len(configs) != 1:
        raise ValueError("The ZIP must contain one Notebook 07 run_config.json.")
    parent = PurePosixPath(configs[0].filename).parent
    names = {PurePosixPath(entry.filename).name: entry for entry in members
             if PurePosixPath(entry.filename).parent == parent}
    return lambda name: TextIOWrapper(archive.open(names[name]), encoding="utf-8", newline="")


def csv_rows(open_file, name):
    with open_file(name) as source:
        yield from csv.DictReader(source)


def optional_number(text):
    if not text:
        return None
    value = float(text)
    if not isfinite(value):
        raise ValueError(f"Expected a finite value, got {text}")
    return value


def charge_at_anchor(row):
    return int(row["landmark_cycle"]) == ANCHOR and row["initializer"] == "charge"

In [2]:
def load_inputs(source):
    with ExitStack() as stack:
        open_file = open_report(source, stack)
        with open_file("run_config.json") as config_file:
            config = json.load(config_file)
        if ANCHOR not in config["landmarks"] or config["eol_soh_pct"] != THRESHOLD:
            raise ValueError("Notebook 07 must include cycle 100 and an 80% SOH threshold.")

        cells = {}
        for row in csv_rows(open_file, "eol_audit.csv"):
            eol = int(row["observed_eol_cycle"]) if row["event_status"] == "observed" else None
            cells[row["cell_id"]] = {
                "rate": int(row["charge_rate_c"]),
                "end": eol if eol is not None else int(row["last_observed_cycle"]),
                "eol": eol,
                "status": row["event_status"],
            }

        test_folds = {}
        training = defaultdict(set)
        for row in csv_rows(open_file, "cell_splits.csv"):
            cell, fold, rate = row["cell_id"], int(row["fold"]), int(row["charge_rate_c"])
            if rate != cells[cell]["rate"]:
                raise ValueError(f"Inconsistent charging rate for {cell}")
            if row["role"] == "test":
                if cell in test_folds:
                    raise ValueError(f"Multiple test folds for {cell}")
                test_folds[cell] = fold
            elif row["role"] == "train":
                training[fold, rate].add(cell)
        folds = set(test_folds.values())
        if set(test_folds) != set(cells) or len(folds) != 5:
            raise ValueError("Expected one held-out fold per cell across five folds.")
        for fold in folds:
            for rate in {cell["rate"] for cell in cells.values()}:
                expected = {cell for cell in cells if cells[cell]["rate"] == rate and test_folds[cell] != fold}
                if training[fold, rate] != expected:
                    raise ValueError(f"Fold {fold} has inconsistent {rate}C training membership")

        starts = {}
        for row in csv_rows(open_file, "rollout_starts.csv"):
            if charge_at_anchor(row):
                cell = row["cell_id"]
                if int(row["fold"]) != test_folds[cell]:
                    raise ValueError(f"Unexpected rollout fold for {cell}")
                starts[cell, row["model"]] = row

        observed = defaultdict(dict)
        saved = defaultdict(dict)
        for row in csv_rows(open_file, "soh_trajectories.csv"):
            if not charge_at_anchor(row):
                continue
            cell, model, cycle = row["cell_id"], row["model"], int(row["cycle"])
            truth = optional_number(row["observed_soh_pct"])
            if cycle in observed[cell] and observed[cell][cycle] != truth:
                raise ValueError(f"Conflicting observations for {cell} at cycle {cycle}")
            observed[cell][cycle] = truth
            saved[cell, model][cycle] = optional_number(row["predicted_soh_pct"])
        for cell, info in cells.items():
            expected = set(range(ANCHOR, info["end"] + 1))
            if set(observed[cell]) != expected:
                raise ValueError(f"Incomplete evaluation timeline for {cell}")
            for model in ORIGINAL_MODELS:
                if (cell, model) not in starts or set(saved[cell, model]) != expected:
                    raise ValueError(f"Missing saved forecast for {cell}, {model}")

        earlier = {}
        for row in csv_rows(open_file, "prior_rul_reference.csv"):
            if charge_at_anchor(row) and row["model"] == EARLIER_RUL:
                cell = row["cell_id"]
                if int(row["fold"]) != test_folds[cell] or int(row["charge_rate_c"]) != cells[cell]["rate"]:
                    raise ValueError(f"Earlier RUL split does not match for {cell}")
                earlier[cell] = optional_number(row["predicted_rul_cycles"])

        profiles = defaultdict(list)
        for row in csv_rows(open_file, "transition_calibration.csv"):
            if row["used"] == "True":
                key = int(row["fold"]), int(row["charge_rate_c"])
                center = float(row["bin_bottom_soh_pct"]) + float(config["bin_width_pp"]) / 2
                profiles[key].append((center, float(row["model_fade_pp_per_cycle"])))
        profiles = {key: tuple(zip(*sorted(profile))) for key, profile in profiles.items()}

    return config, cells, test_folds, training, starts, observed, saved, earlier, profiles

source = locate_input()
config, cells, test_folds, training, starts, observed, saved, earlier, profiles = load_inputs(source)
print(f"Loaded {len(cells)} held-out cells from {source.name}")

Loaded 23 held-out cells from lifeguard_digital_twin_validation_results.zip


## Calibrate late-life timing

At 2C, rescale the original SOH curve to reach 80% at the earlier RUL estimate. At 3C, use the median normalized decline of completed training cells and the mean of the baseline and earlier RUL estimates. Transition after at most 50 cycles or half either predicted remaining life. The curve joins the original forecast at this transition and reaches 80% at the first whole cycle at or after the target.

The earlier RUL for the censored 2C cell is the median remaining life of verified 2C training cells in its fold, the same estimator used for the other 2C cells. An earlier 3C RUL of zero keeps its weight in the average. A missing input falls back to the original curve and is reported.

In [3]:
def interpolate(x, points, values):
    right = bisect_left(points, x)
    if right == 0:
        return values[0]
    if right == len(points):
        return values[-1]
    left = right - 1
    fraction = (x - points[left]) / (points[right] - points[left])
    return values[left] + fraction * (values[right] - values[left])


def crossing(values):
    return next((step for step, value in enumerate(values)
                 if value is not None and value <= THRESHOLD), None)


def original_rollout(initial, profile, steps):
    if initial is None:
        return [None] * (steps + 1)
    values = [initial]
    centers, fades = profile
    for _ in range(steps):
        state = values[-1]
        values.append(max(THRESHOLD, state - interpolate(state, centers, fades))
                      if state > THRESHOLD else state)
    return values


def fit_decay_shape(training_traces):
    curves = []
    for trace in training_traces.values():
        cycles = sorted(trace)
        values = [trace[cycle] for cycle in cycles]
        if len(cycles) < 2 or any(value is None for value in values) or values[0] <= THRESHOLD:
            continue
        start, end = cycles[0], cycles[-1]
        scale = values[0] - THRESHOLD
        damage = [0.0]
        for phase in PHASE_GRID[1:-1]:
            estimate = interpolate(start + phase * (end - start), cycles, values)
            damage.append(min(1.0 - EPS, max(damage[-1], (values[0] - estimate) / scale)))
        curves.append([*damage, 1.0])
    return tuple(median(point) for point in zip(*curves)) if len(curves) >= 2 else None


def calibrate(original, earlier_rul, rate, shape):
    baseline_rul = crossing(original)
    if baseline_rul is None or baseline_rul == 0 or earlier_rul is None or earlier_rul < 0:
        return original, "original_fallback", None, baseline_rul
    if (rate == 2 and earlier_rul == 0) or (rate == 3 and shape is None):
        return original, "original_fallback", None, baseline_rul

    duration = earlier_rul if rate == 2 else (baseline_rul + earlier_rul) / 2
    gate = min(50, baseline_rul // 2, floor(duration / 2))
    gate_soh = original[gate]
    gate_damage = interpolate(gate / duration, PHASE_GRID, shape) if rate == 3 else None
    baseline_steps = range(baseline_rul + 1)
    values = original[:gate + 1]
    for step in range(gate + 1, len(original)):
        if step >= duration:
            value = THRESHOLD
        elif rate == 2:
            mapped_step = gate + (step - gate) / (duration - gate) * (baseline_rul - gate)
            value = interpolate(mapped_step, baseline_steps, original)
        else:
            damage = interpolate(step / duration, PHASE_GRID, shape)
            remaining = (1 - damage) / (1 - gate_damage)
            value = THRESHOLD + (gate_soh - THRESHOLD) * remaining
        values.append(value)
    strategy = "rescaled_fade" if rate == 2 else "training_decay_shape"
    return values, strategy, gate, duration


def make_forecasts():
    completed_training = {
        key: {cell: observed[cell] for cell in sorted(group)
              if cells[cell]["eol"] is not None and cells[cell]["eol"] > ANCHOR}
        for key, group in training.items()
    }
    shapes = {key: fit_decay_shape(traces) for key, traces in completed_training.items() if key[1] == 3}
    results = {}
    for cell, info in cells.items():
        rate = info["rate"]
        key = test_folds[cell], rate
        if cell in completed_training[key]:
            raise AssertionError(f"Held-out cell in the calibration data for {cell}")

        metadata = starts[cell, BASELINE]
        initial = optional_number(metadata["initial_soh_pct"])
        original = original_rollout(initial, profiles[key], config["max_rollout_steps"])
        for cycle, value in saved[cell, BASELINE].items():
            rebuilt = original[cycle - ANCHOR]
            if value != rebuilt and (value is None or rebuilt is None or not isclose(value, rebuilt, abs_tol=EPS)):
                raise ValueError(f"Could not reproduce Notebook 07 for {cell}, cycle {cycle}")
        if crossing(original) != optional_number(metadata["predicted_rul_cycles"]):
            raise ValueError(f"Could not reproduce Notebook 07 crossing for {cell}")

        prior = earlier.get(cell)
        if rate == 2 and completed_training[key]:
            train_median = median(cells[train]["eol"] - ANCHOR for train in completed_training[key])
            if prior is not None and not isclose(prior, train_median, abs_tol=EPS):
                raise ValueError(f"Earlier 2C RUL does not match the training-fold median for {cell}")
            prior = train_median

        values, strategy, gate, duration = calibrate(original, prior, rate, shapes.get(key))
        predicted_rul = crossing(values)
        if values[0] != initial:
            raise AssertionError(f"Calibration changed the starting SOH for {cell}")
        if any(left is not None and right > left + EPS for left, right in zip(values, values[1:])):
            raise AssertionError(f"SOH increased along the calibrated curve for {cell}")
        if gate is not None:
            expected_crossing = next((step for step in range(gate + 1, len(values))
                                      if step >= duration), None)
            if values[:gate + 1] != original[:gate + 1] or predicted_rul != expected_crossing:
                raise AssertionError(f"Incorrect forecast transition or EOL for {cell}")

        results[cell] = {
            "values": values,
            "strategy": strategy,
            "gate": gate,
            "duration": duration,
            "prior": prior,
            "rul": predicted_rul,
            "status": ("initial_soh_unavailable" if initial is None else
                       "no_crossing_by_600" if predicted_rul is None else "crossed"),
        }
    return results


forecasts = make_forecasts()
print(f"Reproduced and calibrated {len(forecasts)} held-out rollouts.")

Reproduced and calibrated 23 held-out rollouts.


In [4]:
def average(values):
    return mean(values) if values else None


def maximum(values):
    return max(values) if values else None


def score():
    case_rows, soh_rows, rul_rows = [], [], []
    metrics = {}
    for rate in sorted({cell["rate"] for cell in cells.values()}):
        cohort = [cell for cell, info in cells.items() if info["rate"] == rate]
        events = [cell for cell in cohort if cells[cell]["eol"] is not None]
        curves = {model: {cell: saved[cell, model] for cell in cohort} for model in ORIGINAL_MODELS}
        curves[CANDIDATE] = {
            cell: {cycle: forecasts[cell]["values"][cycle - ANCHOR] for cycle in observed[cell]}
            for cell in cohort
        }
        windows = {
            **{f"horizon_{h}": [(cell, ANCHOR + h) for cell in cohort] for h in HORIZONS},
            "last_20_cycles": [(cell, cycle) for cell in events for cycle in
                               range(max(ANCHOR + 1, cells[cell]["eol"] - TAIL_CYCLES + 1),
                                     cells[cell]["eol"] + 1)],
            "observed_eol_cycle": [(cell, cells[cell]["eol"]) for cell in events],
        }
        for window, points in windows.items():
            targets = [(cell, cycle) for cell, cycle in points if observed[cell].get(cycle) is not None]
            paired = [(cell, cycle) for cell, cycle in targets
                      if all(curves[model][cell].get(cycle) is not None for model in SOH_MODELS)]
            for model in SOH_MODELS:
                per_cell = defaultdict(list)
                for cell, cycle in paired:
                    per_cell[cell].append(abs(curves[model][cell][cycle] - observed[cell][cycle]))
                errors = [error for group in per_cell.values() for error in group]
                item = {
                    "charge_rate_c": rate, "landmark_cycle": ANCHOR, "initializer": "charge",
                    "model": model, "window": window, "recorded_cycles": len(targets),
                    "paired_cycles": len(paired), "paired_test_cells": len(per_cell),
                    "mae_pp": average(errors), "worst_error_pp": maximum(errors),
                }
                metrics[rate, model, window] = item
                soh_rows.append(item)
                if window == "last_20_cycles":
                    for cell in events:
                        event = cells[cell]["eol"]
                        error = (abs(curves[model][cell][event] - observed[cell][event])
                                 if (cell, event) in paired else None)
                        case_rows.append({
                            "cell_id": cell, "charge_rate_c": rate, "fold": test_folds[cell],
                            "model": model, "observed_eol_cycle": event,
                            "last_20_scored_cycles": len(per_cell[cell]),
                            "last_20_mae_pp": average(per_cell[cell]),
                            "last_20_worst_error_pp": maximum(per_cell[cell]),
                            "observed_eol_soh_error_pp": error,
                        })

        for model in (*SOH_MODELS, EARLIER_RUL):
            if model == CANDIDATE:
                predicted = {cell: forecasts[cell]["rul"] for cell in events}
            elif model == EARLIER_RUL:
                predicted = {cell: earlier.get(cell) for cell in events}
            else:
                predicted = {cell: optional_number(starts[cell, model]["predicted_rul_cycles"]) for cell in events}
            errors = {cell: prediction - (cells[cell]["eol"] - ANCHOR)
                      for cell, prediction in predicted.items() if prediction is not None}
            absolute = [abs(error) for error in errors.values()]
            rul_rows.append({
                "charge_rate_c": rate, "landmark_cycle": ANCHOR, "initializer": "charge",
                "model": model, "observed_test_cells": len(events), "predicted_test_cells": len(errors),
                "mae_cycles": average(absolute), "worst_error_cycles": maximum(absolute),
                "median_signed_error_cycles": median(errors.values()) if errors else None,
                "over_20_cycles_early": sum(error < -20 for error in errors.values()),
                "over_20_cycles_late": sum(error > 20 for error in errors.values()),
            })
            for case in case_rows:
                if case["charge_rate_c"] == rate and case["model"] == model:
                    cell = case["cell_id"]
                    case["predicted_rul_cycles"] = predicted[cell]
                    case["rul_signed_error_cycles"] = errors.get(cell)

    decisions = []
    for rate in sorted({cell["rate"] for cell in cells.values()}):
        best = min(ORIGINAL_MODELS, key=lambda model: metrics[rate, model, "last_20_cycles"]["mae_pp"])
        tail = metrics[rate, CANDIDATE, "last_20_cycles"]
        best_tail = metrics[rate, best, "last_20_cycles"]
        differences = {}
        for h in (10, 50):
            base_metric = metrics[rate, BASELINE, f"horizon_{h}"]
            new_metric = metrics[rate, CANDIDATE, f"horizon_{h}"]
            differences[h] = (new_metric["mae_pp"] - base_metric["mae_pp"]
                              if new_metric["mae_pp"] is not None else None)
        full_coverage = all(
            metrics[rate, CANDIDATE, window]["paired_cycles"] == metrics[rate, CANDIDATE, window]["recorded_cycles"]
            for window in ("last_20_cycles", "horizon_10", "horizon_50")
        )
        approved = (full_coverage and tail["mae_pp"] is not None and tail["mae_pp"] < best_tail["mae_pp"]
                    and all(delta is not None and delta <= HORIZON_TOLERANCE_PP for delta in differences.values()))
        decisions.append({
            "charge_rate_c": rate, "landmark_cycle": ANCHOR, "initializer": "charge", "model": CANDIDATE,
            "best_original_tail_model": best, "best_original_tail_mae_pp": best_tail["mae_pp"],
            "candidate_tail_mae_pp": tail["mae_pp"], "delta_horizon_10_mae_pp": differences[10],
            "delta_horizon_50_mae_pp": differences[50], "full_coverage": full_coverage, "approved": approved,
        })
    return soh_rows, rul_rows, case_rows, decisions


soh_rows, rul_rows, case_rows, decisions = score()

In [5]:
def dashboard_rows():
    trajectories, rollout_starts = [], []
    for cell, info in cells.items():
        result = forecasts[cell]
        common = {"cell_id": cell, "charge_rate_c": info["rate"], "fold": test_folds[cell],
                  "landmark_cycle": ANCHOR, "initializer": "charge", "model": CANDIDATE}
        rollout_starts.append({
            **common, "initial_soh_pct": result["values"][0], "event_status": info["status"],
            "observed_eol_cycle": info["eol"], "predicted_rul_cycles": result["rul"],
            "forecast_status": result["status"], "strategy": result["strategy"],
            "earlier_rul_cycles": result["prior"], "transition_horizon_cycles": result["gate"],
            "target_duration_cycles": result["duration"],
        })
        for cycle, value in sorted(observed[cell].items()):
            horizon = cycle - ANCHOR
            trajectories.append({
                **common, "horizon_cycles": horizon, "cycle": cycle,
                "observed_soh_pct": value, "predicted_soh_pct": result["values"][horizon],
            })
    return trajectories, rollout_starts


def format_metric(value):
    return f"{value:.3f}" if value is not None else "—"


def make_summary():
    lines = [
        "# XJTU fixed cycle-100 SOH forecast",
        "",
        f"{len(cells)} held-out cells across five folds. The forecast uses charge information through cycle 100.",
        "",
        "SOH columns are mean or maximum absolute error in percentage points. RUL MAE is in cycles.",
        "",
        "| Rate | Model | +10 MAE | +50 MAE | Last 20 MAE | Last 20 worst | SOH MAE at EOL | RUL MAE |",
        "| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |",
    ]
    lookup = {(r["charge_rate_c"], r["model"], r["window"]): r for r in soh_rows}
    rul = {(r["charge_rate_c"], r["model"]): r for r in rul_rows}
    for decision in decisions:
        rate = decision["charge_rate_c"]
        for model in SOH_MODELS:
            tail = lookup[rate, model, "last_20_cycles"]
            values = (
                lookup[rate, model, "horizon_10"]["mae_pp"], lookup[rate, model, "horizon_50"]["mae_pp"],
                tail["mae_pp"], tail["worst_error_pp"], lookup[rate, model, "observed_eol_cycle"]["mae_pp"],
                rul[rate, model]["mae_cycles"],
            )
            lines.append("| " + " | ".join((f"{rate}C", model, *map(format_metric, values))) + " |")
    lines.extend(["", "## Dashboard benchmark decision", ""])
    for decision in decisions:
        rate = decision["charge_rate_c"]
        state = "passed" if decision["approved"] else "did not pass"
        lines.append(f"- **{rate}C {state}.** Last-20 MAE "
                     f"{decision['best_original_tail_mae_pp']:.3f} → {decision['candidate_tail_mae_pp']:.3f} pp "
                     f"against the best original model. Earlier RUL benchmark MAE is "
                     f"{format_metric(rul[rate, EARLIER_RUL]['mae_cycles'])} cycles.")
    lines.extend([
        "",
        "The gate requires lower last-20 MAE than the best Notebook 07 model and at most 0.05 pp higher "
        "+10 and +50 MAE than the original SOH-dependent model, with full paired coverage.",
        "",
        "Last-20 and EOL scores require a verified crossing. The right-censored cell contributes "
        "to horizon scores when the target cycle was recorded. A forecast that reaches 80% early "
        "can show a small SOH error at observed EOL, so the RUL timing errors are reported separately.",
        "",
        "The two calibration methods were chosen after inspecting the same XJTU cohort. "
        "Held-out cell scores here measure within-cohort performance.",
    ])
    return "\n".join(lines) + "\n"


def write_csv(path, records):
    with path.open("w", encoding="utf-8", newline="") as output:
        writer = csv.DictWriter(output, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)


output_root = Path(OUTPUT_ROOT).expanduser().resolve() if OUTPUT_ROOT is not None else source.parent
output = output_root / OUTPUT_NAME
output.mkdir(parents=True, exist_ok=True)
trajectories, rollout_starts = dashboard_rows()
horizon_rows = [
    {"charge_rate_c": row["charge_rate_c"], "landmark_cycle": ANCHOR, "initializer": "charge",
     "model": CANDIDATE, "horizon_cycles": int(row["window"].split("_")[-1]),
     "paired_test_cells": row["paired_test_cells"], "paired_mae_pp": row["mae_pp"]}
    for row in soh_rows if row["model"] == CANDIDATE and row["window"].startswith("horizon_")
]
tables = {
    "soh_trajectories.csv": trajectories,
    "rollout_starts.csv": rollout_starts,
    "soh_horizon_metrics.csv": horizon_rows,
    "soh_benchmark.csv": soh_rows,
    "rul_metrics.csv": rul_rows,
    "cell_metrics.csv": case_rows,
    "model_selection.csv": decisions,
}
for name, records in tables.items():
    write_csv(output / name, records)
summary = make_summary()
(output / "summary.md").write_text(summary, encoding="utf-8")
run = {
    "input": source.name, "input_sha256": hashlib.sha256(source.read_bytes()).hexdigest() if source.is_file() else None,
    "landmark_cycle": ANCHOR, "eol_soh_pct": THRESHOLD, "horizons": HORIZONS, "tail_cycles": TAIL_CYCLES,
    "horizon_tolerance_pp": HORIZON_TOLERANCE_PP, "folds": sorted(set(test_folds.values())), "cells": len(cells),
}
(output / "run_config.json").write_text(json.dumps(run, indent=2) + "\n", encoding="utf-8")
archive = output_root / f"{OUTPUT_NAME}.zip"
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for name in (*tables, "summary.md", "run_config.json"):
        bundle.write(output / name, name)

try:
    from IPython.display import Markdown, display
except ImportError:
    print(summary)
else:
    display(Markdown(summary))
print(f"Saved {archive}")

# XJTU fixed cycle-100 SOH forecast

23 held-out cells across five folds. The forecast uses charge information through cycle 100.

SOH columns are mean or maximum absolute error in percentage points. RUL MAE is in cycles.

| Rate | Model | +10 MAE | +50 MAE | Last 20 MAE | Last 20 worst | SOH MAE at EOL | RUL MAE |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 2C | persistence | 0.057 | 0.491 | 16.265 | 18.510 | 17.763 | — |
| 2C | constant_fade | 0.528 | 1.978 | 1.911 | 3.910 | 2.892 | 55.000 |
| 2C | soh_dependent | 0.192 | 0.526 | 1.144 | 3.400 | 0.242 | 31.286 |
| 2C | soh_dependent_with_history | 0.266 | 1.330 | 1.305 | 3.400 | 0.214 | 114.143 |
| 2C | late_life_calibrated | 0.192 | 0.526 | 0.743 | 3.067 | 0.688 | 9.000 |
| 3C | persistence | 0.712 | 2.591 | 12.706 | 18.332 | 15.552 | — |
| 3C | constant_fade | 0.368 | 1.596 | 5.541 | 10.034 | 7.899 | 151.267 |
| 3C | soh_dependent | 0.241 | 0.396 | 2.627 | 8.758 | 2.388 | 23.600 |
| 3C | soh_dependent_with_history | 0.201 | 0.529 | 2.293 | 6.700 | 1.633 | 34.933 |
| 3C | late_life_calibrated | 0.241 | 0.422 | 2.187 | 7.120 | 2.136 | 20.800 |

## Dashboard benchmark decision

- **2C passed.** Last-20 MAE 1.144 → 0.743 pp against the best original model. Earlier RUL benchmark MAE is 9.000 cycles.
- **3C passed.** Last-20 MAE 2.293 → 2.187 pp against the best original model. Earlier RUL benchmark MAE is 21.937 cycles.

The gate requires lower last-20 MAE than the best Notebook 07 model and at most 0.05 pp higher +10 and +50 MAE than the original SOH-dependent model, with full paired coverage.

Last-20 and EOL scores require a verified crossing. The right-censored cell contributes to horizon scores when the target cycle was recorded. A forecast that reaches 80% early can show a small SOH error at observed EOL, so the RUL timing errors are reported separately.

The two calibration methods were chosen after inspecting the same XJTU cohort. Held-out cell scores here measure within-cohort performance.


Saved /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_fixed_anchor_soh_results.zip
